In [1]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_URL = "https://api.openai.com/v1/audio/speech"
OPENAI_MODEL = "gpt-4o-mini-tts"
# voice options https://platform.openai.com/docs/guides/text-to-speech#voice-options
OPENAI_VOICE = "marin"
OPENAI_FORMAT = "mp3"  # mp3, wav, opus, aac, flac, pcm
OPENAI_INSTRUCTIONS = None  # e.g., "Speak in a cheerful and positive tone."


## OpenAI (TTS)

Reference: https://platform.openai.com/docs/guides/text-to-speech


In [2]:
def openai_tts_bytes(text: str, output_path: str | None = "outputs/openai_tts.mp3") -> bytes:
    if not OPENAI_API_KEY:
        raise ValueError("Missing OPENAI_API_KEY in notebooks/.env")

    payload = {
        "model": OPENAI_MODEL,
        "input": text,
        "voice": OPENAI_VOICE,
        "response_format": OPENAI_FORMAT,
    }
    if OPENAI_INSTRUCTIONS:
        payload["instructions"] = OPENAI_INSTRUCTIONS

    auth = OPENAI_API_KEY.strip().strip("'")
    if not auth.startswith("Bearer "):
        auth = f"Bearer {auth}"

    headers = {
        "Authorization": auth,
        "Content-Type": "application/json",
    }

    r = requests.post(OPENAI_URL, json=payload, headers=headers, timeout=60)
    r.raise_for_status()

    audio = r.content
    if output_path:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(audio)
    return audio


In [3]:
audio = openai_tts_bytes("Hello from OpenAI. This is a quick TTS smoke test.")


In [4]:
len(audio)


78720

In [5]:
display(Audio(audio))
